# Import Libraries

In [15]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.io import output_notebook
import pandas as pd

output_notebook()

Loading BokehJS ...

### Important Terms

1. Sharpe Ratio → Return earned per unit of risk; higher is better

2. Correlation → How two assets move together; lower (or negative) is better for diversification

3. Beta (vs Nifty 50) → Sensitivity of a stock to market movements; depends (≈1 market-like, <1 safer, >1 riskier)

4. Max Drawdown → Largest peak-to-bottom loss; lower is better

5. SMA (Simple Moving Average) → Average price over a period to show trend; no “better”, it’s a tool (price above SMA = bullish, below = bearish)


# Dataset loading and Cleaning 

In [ ]:
stocks = ["HDFCBANK.NS", "ICICIBANK.NS", "SBIN.NS", "KOTAKBANK.NS", "AXISBANK.NS","TCS.NS", "INFY.NS", "WIPRO.NS", "HCLTECH.NS", "TECHM.NS",
            "SUNPHARMA.NS", "DRREDDY.NS", "CIPLA.NS", "DIVISLAB.NS", "APOLLOHOSP.NS"
]
benchmark= "^NSEI"

all_tickers = stocks + [benchmark]

# ==== FETCH DATA ====
data = yf.download(
    all_tickers,
    start="2023-01-01",
    end="2024-12-31",
    auto_adjust=True, 
    progress=False
)

print(data.head())


In [36]:
# Get only Adjusted Close prices
prices = data['Close']   # since auto_adjust=True, Close = Adj Close

# Check missing values
# print(prices.isnull().sum())

# Fill small gaps
prices = prices.ffill()

# Drop any remaining missing values
prices = prices.dropna()

returns = prices.pct_change().dropna()

annual_return = returns.mean() * 252

volatility = returns.std() * (252**0.5)

risk_free_rate = 0.065
sharpe = (annual_return - risk_free_rate) / volatility

cov_matrix = returns.cov()
beta = cov_matrix.loc[stocks, "^NSEI"] / cov_matrix.loc["^NSEI", "^NSEI"]

max_drawdown = {}

for stock in stocks:
    cum = (1 + returns[stock]).cumprod()
    peak = cum.cummax()
    drawdown = (cum - peak) / peak
    max_drawdown[stock] = drawdown.min()

metrics_df = pd.DataFrame({
    "Return": annual_return,
    "Volatility": volatility,
    "Sharpe": sharpe,
    "Beta": pd.Series(beta),
    "Max Drawdown": pd.Series(max_drawdown)
})

metrics_df['Type'] = metrics_df['Beta'].apply(
    lambda x: "Aggressive" if x > 1 else "Defensive"
)

metrics_df['Sharpe Rank'] = metrics_df['Sharpe'].rank(ascending=False)

sector_map = {
    "HDFCBANK.NS": "Banking",
    "TCS.NS": "IT"
}
metrics_df['Sector'] = metrics_df.index.map(sector_map)

def interpret(row):
    if row['Sharpe'] > 1:
        return "Efficient"
    elif row['Volatility'] > 0.3:
        return "High Risk"
    else:
        return "Moderate"

metrics_df['Insight'] = metrics_df.apply(interpret, axis=1)

print(metrics_df)

                 Return  Volatility    Sharpe      Beta  Max Drawdown  \
APOLLOHOSP.NS  0.282127    0.217577  0.997933  0.563492     -0.148744   
AXISBANK.NS    0.090783    0.219231  0.117606  1.060098     -0.187770   
CIPLA.NS       0.218251    0.240903  0.636152  0.332154     -0.207654   
DIVISLAB.NS    0.342583    0.262767  1.056384  0.452973     -0.214458   
DRREDDY.NS     0.276510    0.194645  1.086644  0.429148     -0.156136   
HCLTECH.NS     0.382456    0.213823  1.484666  0.787122     -0.216645   
HDFCBANK.NS    0.077817    0.198171  0.064675  1.039062     -0.199138   
ICICIBANK.NS   0.211509    0.183396  0.798869  0.969437     -0.093360   
INFY.NS        0.171573    0.230859  0.461635  0.901684     -0.243414   
KOTAKBANK.NS  -0.002383    0.204081 -0.330177  0.863379     -0.231527   
SBIN.NS        0.181923    0.255895  0.456917  1.391631     -0.174804   
SUNPHARMA.NS   0.353453    0.173917  1.658571  0.472544     -0.122387   
TCS.NS         0.172376    0.196971  0.545134  0.74

# Scatter Plot Risk Vs Return

In [39]:
# ==== IMPORTS ====
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool, LabelSet
from bokeh.io import output_notebook
import pandas as pd

output_notebook()

# ==== PREP DATA ====
df_plot = metrics_df.copy()

# Remove NIFTY (not needed in scatter)
df_plot = df_plot.drop("^NSEI", errors="ignore")

# Reset index → VERY IMPORTANT
df_plot = df_plot.reset_index().rename(columns={"index": "Stock"})

# Ensure numeric columns
cols = ["Return", "Volatility", "Sharpe", "Beta"]
for col in cols:
    df_plot[col] = pd.to_numeric(df_plot[col], errors="coerce")

# Drop only essential NaNs
df_plot = df_plot.dropna(subset=["Return", "Volatility"])

# Optional: color based on Sharpe
df_plot["Color"] = df_plot["Sharpe"].apply(
    lambda x: "green" if x > 1 else ("orange" if x > 0.5 else "red")
)

# ==== DATA SOURCE ====
source = ColumnDataSource(df_plot)

# ==== PLOT ====
p = figure(
    title="Risk vs Return (Interactive)",
    x_axis_label="Volatility (Risk)",
    y_axis_label="Return",
    width=800,
    height=500,
    tools="pan,wheel_zoom,box_zoom,reset"
)

# Scatter points
p.scatter(
    x="Volatility",
    y="Return",
    size=10,
    color="Color",
    source=source
)

# ==== HOVER TOOL ====
hover = HoverTool(tooltips=[
    ("Stock", "@Stock"),
    ("Return", "@Return{0.00%}"),
    ("Volatility", "@Volatility{0.00%}"),
    ("Sharpe", "@Sharpe{0.00}"),
    ("Beta", "@Beta{0.00}")
])
p.add_tools(hover)

# ==== LABELS ====
labels = LabelSet(
    x='Volatility',
    y='Return',
    text='Stock',
    source=source,
    text_font_size="8pt"
)
p.add_layout(labels)

# ==== SHOW ====
show(p)

Loading BokehJS ...

# Heatmap Correlation

In [40]:
# ==== IMPORTS ====
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool, ColorBar
from bokeh.transform import linear_cmap
from bokeh.palettes import RdBu
from bokeh.io import output_notebook
import pandas as pd

output_notebook()

# ==== PREP DATA ====
corr = returns.corr()

# Fix pandas naming bug
corr.columns.name = None
corr.index.name = None

# Convert to long format
corr_df = corr.stack().reset_index()
corr_df.columns = ["Stock1", "Stock2", "Correlation"]

# Remove diagonal (self-correlation)
corr_df = corr_df[corr_df["Stock1"] != corr_df["Stock2"]]

# ==== DATA SOURCE ====
source = ColumnDataSource(corr_df)

# ==== COLOR MAPPING ====
mapper = linear_cmap(
    field_name="Correlation",
    palette=RdBu[11],
    low=-1,
    high=1
)

# ==== PLOT ====
p = figure(
    title="Correlation Heatmap",
    x_range=list(corr.columns),
    y_range=list(corr.columns),
    width=700,
    height=700,
    tools="pan,wheel_zoom,box_zoom,reset"
)

# Rectangles
p.rect(
    x="Stock1",
    y="Stock2",
    width=1,
    height=1,
    source=source,
    fill_color=mapper,
    line_color=None
)

# ==== HOVER TOOL ====
hover = HoverTool(tooltips=[
    ("Stock 1", "@Stock1"),
    ("Stock 2", "@Stock2"),
    ("Correlation", "@Correlation{0.00}")
])
p.add_tools(hover)

# ==== COLOR BAR ====
color_bar = ColorBar(color_mapper=mapper["transform"], width=8)
p.add_layout(color_bar, 'right')

# Rotate labels for readability
p.xaxis.major_label_orientation = 1.2

# ==== SHOW ====
show(p)

Loading BokehJS ...

In [43]:
# ==== FINAL SUMMARY TABLE ====

summary_df = metrics_df.copy()

# Remove NIFTY (optional)
summary_df = summary_df.drop("^NSEI", errors="ignore")

# Reset index → make Stock a column
summary_df = summary_df.reset_index().rename(columns={"index": "Stock"})

# Add Crash Loss
summary_df["Crash_Loss_%"] = summary_df["Beta"] * (-10)

# Select & order columns
summary_df = summary_df[[
    "Stock",
    "Return",
    "Volatility",
    "Sharpe",
    "Beta",
    "Max Drawdown",
    "Crash_Loss_%"
]]

# Round for clean presentation
summary_df = summary_df.round(4)

# Sort by Sharpe (best first)
summary_df = summary_df.sort_values("Sharpe", ascending=False)

# Display
print(summary_df)

# Save (optional)
summary_df.to_csv("data/summary_table.csv", index=False)

            Stock  Return  Volatility  Sharpe    Beta  Max Drawdown  \
11   SUNPHARMA.NS  0.3535      0.1739  1.6586  0.4725       -0.1224   
5      HCLTECH.NS  0.3825      0.2138  1.4847  0.7871       -0.2166   
13       TECHM.NS  0.3453      0.2514  1.1150  0.8729       -0.1622   
4      DRREDDY.NS  0.2765      0.1946  1.0866  0.4291       -0.1561   
3     DIVISLAB.NS  0.3426      0.2628  1.0564  0.4530       -0.2145   
0   APOLLOHOSP.NS  0.2821      0.2176  0.9979  0.5635       -0.1487   
7    ICICIBANK.NS  0.2115      0.1834  0.7989  0.9694       -0.0934   
14       WIPRO.NS  0.2559      0.2417  0.7896  1.0728       -0.1953   
2        CIPLA.NS  0.2183      0.2409  0.6362  0.3322       -0.2077   
12         TCS.NS  0.1724      0.1970  0.5451  0.7472       -0.1317   
8         INFY.NS  0.1716      0.2309  0.4616  0.9017       -0.2434   
10        SBIN.NS  0.1819      0.2559  0.4569  1.3916       -0.1748   
1     AXISBANK.NS  0.0908      0.2192  0.1176  1.0601       -0.1878   
6     

## Exporting cleaned dataset

In [42]:
# ==== IMPORT ====
import os
import pandas as pd

# ==== COPY METRICS ====
final_df = metrics_df.copy()

# Remove NIFTY (optional but cleaner)
final_df = final_df.drop("^NSEI", errors="ignore")

# ==== ADD STOCK COLUMN ====
final_df = final_df.reset_index().rename(columns={"index": "Stock"})

# ==== ADD CRASH TEST COLUMN ====
final_df["Crash_Loss_%"] = final_df["Beta"] * (-10)

# ==== ROUND VALUES (clean look) ====
final_df = final_df.round(4)

# ==== CREATE FOLDER ====
folder = "data"
os.makedirs(folder, exist_ok=True)

# ==== EXPORT ====
file_path = os.path.join(folder, "final_stock_analysis.csv")
final_df.to_csv(file_path, index=False)

# ==== CONFIRM ====
print("File saved at:", os.path.abspath(file_path))
print("\nPreview:")
print(final_df.head())

File saved at: c:\Users\singh\Desktop\McGills\Hackathons\FedVal\fedeval\data\final_stock_analysis.csv

Preview:
           Stock  Return  Volatility  Sharpe    Beta  Max Drawdown  \
0  APOLLOHOSP.NS  0.2821      0.2176  0.9979  0.5635       -0.1487   
1    AXISBANK.NS  0.0908      0.2192  0.1176  1.0601       -0.1878   
2       CIPLA.NS  0.2183      0.2409  0.6362  0.3322       -0.2077   
3    DIVISLAB.NS  0.3426      0.2628  1.0564  0.4530       -0.2145   
4     DRREDDY.NS  0.2765      0.1946  1.0866  0.4291       -0.1561   

         Type  Sharpe Rank Sector    Insight  Crash_Loss_%  
0   Defensive          6.0    NaN   Moderate       -5.6349  
1  Aggressive         14.0    NaN   Moderate      -10.6010  
2   Defensive         10.0    NaN   Moderate       -3.3215  
3   Defensive          5.0    NaN  Efficient       -4.5297  
4   Defensive          4.0    NaN  Efficient       -4.2915  


# Technical Signal Dashboard

In [ ]:
# ==== SMA CALCULATION ====

sma50 = prices.rolling(window=50).mean()
sma200 = prices.rolling(window=200).mean()

In [47]:
# ==== SIGNAL TABLE ====

signal_data = []

for stock in prices.columns:
    if stock == "^NSEI":
        continue

    latest_sma50 = sma50[stock].iloc[-1]
    latest_sma200 = sma200[stock].iloc[-1]

    if latest_sma50 > latest_sma200:
        signal = "Golden Cross (Bullish)"
    else:
        signal = "Death Cross (Bearish)"

    signal_data.append({
        "Stock": stock,
        "SMA50": latest_sma50,
        "SMA200": latest_sma200,
        "Signal": signal
    })

signal_df = pd.DataFrame(signal_data)

# Clean
signal_df = signal_df.round(4)

print(signal_df)

            Stock      SMA50     SMA200                  Signal
0   APOLLOHOSP.NS  7050.7052  6559.5479  Golden Cross (Bullish)
1     AXISBANK.NS  1144.5458  1160.8014   Death Cross (Bearish)
2        CIPLA.NS  1487.4268  1492.8654   Death Cross (Bearish)
3     DIVISLAB.NS  5901.8108  4838.5983  Golden Cross (Bullish)
4      DRREDDY.NS  1263.8852  1267.8779   Death Cross (Bearish)
5      HCLTECH.NS  1792.2797  1558.6834  Golden Cross (Bullish)
6     HDFCBANK.NS   876.1511   804.5476  Golden Cross (Bullish)
7    ICICIBANK.NS  1282.1707  1188.1721  Golden Cross (Bullish)
8         INFY.NS  1825.7190  1648.6318  Golden Cross (Bullish)
9    KOTAKBANK.NS   352.0974   353.3581   Death Cross (Bearish)
10        SBIN.NS   812.2739   795.1305  Golden Cross (Bullish)
11   SUNPHARMA.NS  1789.6400  1661.1147  Golden Cross (Bullish)
12         TCS.NS  3988.2311  3898.4550  Golden Cross (Bullish)
13       TECHM.NS  1665.6111  1439.6931  Golden Cross (Bullish)
14       WIPRO.NS   271.7960   244.0608 

In [52]:
# ==== IMPORTS ====
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.io import output_notebook
import pandas as pd

output_notebook()

# ==== SELECT STOCK ====
stock = "TCS.NS"

# ==== PREP DATA ====
df_chart = pd.DataFrame({
    "Price": prices[stock],
    "SMA50": sma50[stock],
    "SMA200": sma200[stock]
}).dropna()

df_chart = df_chart.reset_index().rename(columns={"index": "Date"})

source = ColumnDataSource(df_chart)

# ==== PLOT ====
p = figure(
    x_axis_type="datetime",
    title=f"{stock} Price with SMA Signals",
    width=800,
    height=400,
    tools="pan,wheel_zoom,box_zoom,reset"
)

# 🔥 Price (blue)
p.line("Date", "Price", source=source, line_width=2, color="blue", legend_label="Price")

# 🔥 SMA 50 (orange)
p.line("Date", "SMA50", source=source, line_width=2, color="orange", legend_label="SMA 50")

# 🔥 SMA 200 (red)
p.line("Date", "SMA200", source=source, line_width=2, color="red", legend_label="SMA 200")

# ==== HOVER ====
hover = HoverTool(tooltips=[
    ("Date", "@Date{%F}"),
    ("Price", "@Price{0.2f}"),
    ("SMA50", "@SMA50{0.2f}"),
    ("SMA200", "@SMA200{0.2f}")
], formatters={"@Date": "datetime"})

p.add_tools(hover)

# ==== LEGEND ====
p.legend.location = "top_left"
p.legend.click_policy = "hide"  # 🔥 allows toggling lines

# ==== SHOW ====
show(p)

Loading BokehJS ...

In [63]:
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource
import pandas as pd

# ===== SELECT 3 STOCKS =====
stocks = ["HDFCBANK.NS", "TCS.NS", "SUNPHARMA.NS"]

for stock in stocks:
    
    # Prepare dataframe
    df = pd.DataFrame({
        "Price": prices[stock],
        "SMA50": sma50[stock],
        "SMA200": sma200[stock]
    }).dropna()

    df = df.reset_index()

    # ===== SIGNALS =====
    df["Signal"] = 0
    df.loc[df["SMA50"] > df["SMA200"], "Signal"] = 1
    df["Crossover"] = df["Signal"].diff()

    golden = df[df["Crossover"] == 1]
    death = df[df["Crossover"] == -1]

    source = ColumnDataSource(df)

    # ===== PLOT =====
    p = figure(
        title=f"{stock} Price with SMA Signals",
        x_axis_type="datetime",
        width=900,
        height=500,
        tools="pan,wheel_zoom,box_zoom,reset"
    )

    # Price
    p.line("Date", "Price", source=source, line_width=2, legend_label="Price", color="black")

    # SMA lines
    p.line("Date", "SMA50", source=source, line_width=2, color="blue", legend_label="SMA 50")
    p.line("Date", "SMA200", source=source, line_width=2, color="orange", legend_label="SMA 200")

    # Golden Cross (green)
    p.scatter(
        golden["Date"], golden["Price"],
        size=10, color="green", legend_label="Golden Cross"
    )

    # Death Cross (red)
    p.scatter(
        death["Date"], death["Price"],
        size=10, color="red", legend_label="Death Cross"
    )

    p.legend.location = "top_left"

    show(p)

In [49]:
# ==== CROSSOVER DETECTION ====

crossovers = []

for stock in prices.columns:
    if stock == "^NSEI":
        continue

    df = pd.DataFrame({
        "SMA50": sma50[stock],
        "SMA200": sma200[stock]
    }).dropna()

    df["Signal"] = 0
    df.loc[df["SMA50"] > df["SMA200"], "Signal"] = 1

    df["Crossover"] = df["Signal"].diff()

    golden = df[df["Crossover"] == 1]
    death = df[df["Crossover"] == -1]

    crossovers.append({
        "Stock": stock,
        "Golden Crosses": len(golden),
        "Death Crosses": len(death)
    })

crossover_df = pd.DataFrame(crossovers)
print(crossover_df)

            Stock  Golden Crosses  Death Crosses
0   APOLLOHOSP.NS               0              0
1     AXISBANK.NS               0              1
2        CIPLA.NS               0              1
3     DIVISLAB.NS               1              1
4      DRREDDY.NS               0              1
5      HCLTECH.NS               1              1
6     HDFCBANK.NS               1              0
7    ICICIBANK.NS               0              0
8         INFY.NS               1              1
9    KOTAKBANK.NS               1              1
10        SBIN.NS               0              0
11   SUNPHARMA.NS               0              0
12         TCS.NS               0              0
13       TECHM.NS               0              0
14       WIPRO.NS               0              0


# Best and Worst three Stocks

In [54]:
# ==== REMOVE NIFTY (if present) ====
df = metrics_df.drop("^NSEI", errors="ignore")

# ==== BEST 3 (highest Sharpe) ====
best_3 = df.sort_values("Sharpe", ascending=False).head(3)

# ==== WORST 3 (lowest Sharpe) ====
worst_3 = df.sort_values("Sharpe", ascending=True).head(3)

print("=== BEST 3 STOCKS (Highest Sharpe) ===")
print(best_3[["Return", "Volatility", "Sharpe", "Beta", "Max Drawdown"]])

print("\n=== WORST 3 STOCKS (Lowest Sharpe) ===")
print(worst_3[["Return", "Volatility", "Sharpe", "Beta", "Max Drawdown"]])

summary = pd.concat([best_3.assign(Type="Best"), worst_3.assign(Type="Worst")])
print(summary)

=== BEST 3 STOCKS (Highest Sharpe) ===
                Return  Volatility    Sharpe      Beta  Max Drawdown
SUNPHARMA.NS  0.353453    0.173917  1.658571  0.472544     -0.122387
HCLTECH.NS    0.382456    0.213823  1.484666  0.787122     -0.216645
TECHM.NS      0.345313    0.251405  1.114983  0.872880     -0.162151

=== WORST 3 STOCKS (Lowest Sharpe) ===
                Return  Volatility    Sharpe      Beta  Max Drawdown
KOTAKBANK.NS -0.002383    0.204081 -0.330177  0.863379     -0.231527
HDFCBANK.NS   0.077817    0.198171  0.064675  1.039062     -0.199138
AXISBANK.NS   0.090783    0.219231  0.117606  1.060098     -0.187770
                Return  Volatility    Sharpe      Beta  Max Drawdown   Type  \
SUNPHARMA.NS  0.353453    0.173917  1.658571  0.472544     -0.122387   Best   
HCLTECH.NS    0.382456    0.213823  1.484666  0.787122     -0.216645   Best   
TECHM.NS      0.345313    0.251405  1.114983  0.872880     -0.162151   Best   
KOTAKBANK.NS -0.002383    0.204081 -0.330177  0.86337

# Portfolio Construction

In [ ]:
# ==== PORTFOLIO A ====

stocks_all = [s for s in returns.columns if s != "^NSEI"]

n = len(stocks_all)
weights_A = np.array([1/n] * n)

# Portfolio A metrics
data_A = returns[stocks_all]

ret_A = np.dot(data_A.mean()*252, weights_A)
vol_A = np.sqrt(np.dot(weights_A.T, np.dot(data_A.cov()*252, weights_A)))

print("Portfolio A Return:", ret_A)
print("Portfolio A Volatility:", vol_A)



Portfolio A Return: 0.22401003397517066
Portfolio A Volatility: 0.11427040089113154


In [58]:
portfolio_B = ["SUNPHARMA.NS", "HCLTECH.NS", "DRREDDY.NS", "TECHM.NS", "ICICIBANK.NS"]

weights_B = np.array([0.25, 0.20, 0.20, 0.20, 0.15])

data_B = returns[portfolio_B]

ret_B = np.dot(data_B.mean()*252, weights_B)

vol_B = np.sqrt(np.dot(weights_B.T, np.dot(data_B.cov()*252, weights_B)))

rf = 0.065
sharpe_B = (ret_B - rf) / vol_B

print("Return:", ret_B)
print("Volatility:", vol_B)
print("Sharpe:", sharpe_B)

Return: 0.3209451973088415
Volatility: 0.12433974597723815
Sharpe: 2.058434294659853


In [59]:
metrics_df["Crash_Loss_%"] = metrics_df["Beta"] * (-10)

crash_B = np.dot(
    metrics_df.loc[portfolio_B]["Crash_Loss_%"],
    weights_B
)

print("Portfolio B Crash Loss (%):", crash_B)

Portfolio B Crash Loss (%): -6.813815649923329


In [60]:
summary = pd.DataFrame({
    "Portfolio": ["B (Optimized)"],
    "Return": [ret_B],
    "Volatility": [vol_B],
    "Sharpe": [sharpe_B],
    "Crash Loss (%)": [crash_B]
})

print(summary.round(4))

       Portfolio  Return  Volatility  Sharpe  Crash Loss (%)
0  B (Optimized)  0.3209      0.1243  2.0584         -6.8138


In [61]:
import pandas as pd
import numpy as np

# ==== PORTFOLIOS ====
stocks_all = [s for s in returns.columns if s != "^NSEI"]

portfolio_A = stocks_all
portfolio_B = ["SUNPHARMA.NS", "HCLTECH.NS", "DRREDDY.NS", "TECHM.NS", "ICICIBANK.NS"]

# ==== WEIGHTS ====
weights_A = np.array([1/len(portfolio_A)] * len(portfolio_A))
weights_B = np.array([0.25, 0.20, 0.20, 0.20, 0.15])

# ==== RISK FREE RATE ====
rf = 0.065

# ==== FUNCTION ====
def compute_metrics(stocks, weights):
    data = returns[stocks]

    port_return = np.dot(data.mean() * 252, weights)
    port_vol = np.sqrt(np.dot(weights.T, np.dot(data.cov() * 252, weights)))
    sharpe = (port_return - rf) / port_vol

    return port_return, port_vol, sharpe

# ==== CALCULATE ====
ret_A, vol_A, sharpe_A = compute_metrics(portfolio_A, weights_A)
ret_B, vol_B, sharpe_B = compute_metrics(portfolio_B, weights_B)

# ==== SECTOR MAP ====
sector_map = {
    "HDFCBANK.NS": "Banking",
    "ICICIBANK.NS": "Banking",
    "SBIN.NS": "Banking",
    "KOTAKBANK.NS": "Banking",
    "AXISBANK.NS": "Banking",
    "TCS.NS": "IT",
    "INFY.NS": "IT",
    "WIPRO.NS": "IT",
    "HCLTECH.NS": "IT",
    "TECHM.NS": "IT",
    "SUNPHARMA.NS": "Pharma",
    "DRREDDY.NS": "Pharma",
    "CIPLA.NS": "Pharma",
    "DIVISLAB.NS": "Pharma",
    "APOLLOHOSP.NS": "Pharma"
}

# ==== SECTOR EXPOSURE FUNCTION ====
def sector_exposure(stocks, weights):
    exposure = {}

    for s, w in zip(stocks, weights):
        sector = sector_map[s]
        exposure[sector] = exposure.get(sector, 0) + w

    return exposure

sector_A = sector_exposure(portfolio_A, weights_A)
sector_B = sector_exposure(portfolio_B, weights_B)

# Convert sector dict to string (clean for display)
sector_A_str = ", ".join([f"{k}:{round(v*100,1)}%" for k,v in sector_A.items()])
sector_B_str = ", ".join([f"{k}:{round(v*100,1)}%" for k,v in sector_B.items()])

# ==== FINAL COMPARISON TABLE ====
comparison_df = pd.DataFrame({
    "Portfolio": ["A (Equal Weight)", "B (Optimized)"],
    "Annual Return": [ret_A, ret_B],
    "Volatility": [vol_A, vol_B],
    "Sharpe Ratio": [sharpe_A, sharpe_B],
    "Sector Exposure": [sector_A_str, sector_B_str]
})

# ==== CLEAN OUTPUT ====
comparison_df = comparison_df.round(4)

print(comparison_df)

          Portfolio  Annual Return  Volatility  Sharpe Ratio  \
0  A (Equal Weight)         0.2240      0.1143        1.3915   
1     B (Optimized)         0.3209      0.1243        2.0584   

                         Sector Exposure  
0  Pharma:33.3%, Banking:33.3%, IT:33.3%  
1  Pharma:45.0%, IT:40.0%, Banking:15.0%  


In [62]:
# ==== REMOVE NIFTY ====
df = metrics_df.drop("^NSEI", errors="ignore").copy()

# ==== ADD SECTOR COLUMN ====
df["Sector"] = df.index.map(sector_map)

# ==== GROUP BY SECTOR ====
sector_stats = df.groupby("Sector").agg({
    "Sharpe": "mean",
    "Beta": "mean"
}).reset_index()

# ==== CLEAN FORMAT ====
sector_stats = sector_stats.round(3)

print(sector_stats)

    Sector  Sharpe   Beta
0  Banking   0.222  1.065
1       IT   0.879  0.876
2   Pharma   1.087  0.450


# Stress Test for 10% fall in single week

In [65]:

# ===== ASSUMPTION =====
market_drop = -0.10   # -10%

# ===== REMOVE NIFTY =====
df = metrics_df.copy()
df = df.drop("^NSEI", errors="ignore")

# ===== STOCK LEVEL LOSS =====
df["Crash_Loss_%"] = df["Beta"] * market_drop * 100

# ===== PORTFOLIO A (Equal Weight) =====
stocks_A = df.index.tolist()
weights_A = np.array([1/len(stocks_A)] * len(stocks_A))

loss_A = np.dot(weights_A, df["Crash_Loss_%"])

# ===== PORTFOLIO B (Your selected stocks) =====
portfolio_B = ["SUNPHARMA.NS", "HCLTECH.NS", "DRREDDY.NS", "TECHM.NS", "ICICIBANK.NS"]
weights_B = np.array([0.25, 0.20, 0.20, 0.20, 0.15])

df_B = df.loc[portfolio_B]

loss_B = np.dot(weights_B, df_B["Crash_Loss_%"])

# ===== MOST EXPOSED & SAFEST =====
most_exposed = df["Crash_Loss_%"].idxmin()   # most negative
safest = df["Crash_Loss_%"].idxmax()         # least negative

# ===== OUTPUT =====
print("\n=== STOCK LEVEL CRASH IMPACT ===")
print(df[["Beta", "Crash_Loss_%"]].sort_values("Crash_Loss_%"))

print("\n=== PORTFOLIO IMPACT ===")
print(f"Portfolio A Loss: {loss_A:.2f}%")
print(f"Portfolio B Loss: {loss_B:.2f}%")

print("\n=== INSIGHTS ===")
print(f"Most Exposed Stock: {most_exposed}")
print(f"Safest Stock: {safest}")


=== STOCK LEVEL CRASH IMPACT ===
                   Beta  Crash_Loss_%
SBIN.NS        1.391631    -13.916315
WIPRO.NS       1.072815    -10.728146
AXISBANK.NS    1.060098    -10.600982
HDFCBANK.NS    1.039062    -10.390625
ICICIBANK.NS   0.969437     -9.694372
INFY.NS        0.901684     -9.016839
TECHM.NS       0.872880     -8.728799
KOTAKBANK.NS   0.863379     -8.633792
HCLTECH.NS     0.787122     -7.871215
TCS.NS         0.747187     -7.471870
APOLLOHOSP.NS  0.563492     -5.634917
SUNPHARMA.NS   0.472544     -4.725440
DIVISLAB.NS    0.452973     -4.529725
DRREDDY.NS     0.429148     -4.291485
CIPLA.NS       0.332154     -3.321541

=== PORTFOLIO IMPACT ===
Portfolio A Loss: -7.97%
Portfolio B Loss: -6.81%

=== INSIGHTS ===
Most Exposed Stock: SBIN.NS
Safest Stock: CIPLA.NS


# ML MODEL

In [66]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# ===== PREPARE DATA =====

data = returns.copy()

# ===== FEATURES =====
features = pd.DataFrame(index=data.index)

# rolling features
features["mean_5"] = data.mean(axis=1)
features["vol_5"] = data.std(axis=1)

# ===== TARGET =====
# next day return (shift)
future_return = data.shift(-1)

# label: outperform (above median)
median_return = future_return.median(axis=1)

target = (future_return.gt(median_return, axis=0)).astype(int)

# flatten dataset (stock-wise)
X = []
y = []

for col in data.columns:
    df_temp = pd.DataFrame({
        "mean_5": features["mean_5"],
        "vol_5": features["vol_5"],
        "target": target[col]
    }).dropna()

    X.append(df_temp[["mean_5", "vol_5"]])
    y.append(df_temp["target"])

X = pd.concat(X)
y = pd.concat(y)

# ===== TRAIN TEST SPLIT (IMPORTANT: TIME SPLIT) =====
split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# ===== MODEL =====
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ===== EVALUATION =====
pred = model.predict(X_test)

print("\n=== MODEL PERFORMANCE ===")
print(classification_report(y_test, pred))

# ===== FEATURE IMPORTANCE =====
importance = pd.Series(model.feature_importances_, index=X.columns)
print("\n=== FEATURE IMPORTANCE ===")
print(importance)


=== MODEL PERFORMANCE ===
              precision    recall  f1-score   support

           0       0.28      0.27      0.28       783
           1       0.28      0.29      0.28       782

    accuracy                           0.28      1565
   macro avg       0.28      0.28      0.28      1565
weighted avg       0.28      0.28      0.28      1565


=== FEATURE IMPORTANCE ===
mean_5    0.49805
vol_5     0.50195
dtype: float64
